# 03 - Submission
**Tujuan:** pembuatan file submission dari model final (`models/pipeline_model.pkl`) dan pengecekan kesesuaian format dengan `sample_submission.csv`.

In [3]:
# Setup
import sys
sys.path.append("..")   # root repo ditambahkan ke path

import pandas as pd
from scripts.config import *
from scripts.preprocess import load_data
from scripts.predict import load_model, make_submission

In [4]:
# Load data test dan model final
test = load_data(TEST_PATH)
model = load_model()
print("Test:", test.shape)

Test: (228629, 16)


In [5]:
# Prediksi
submission = make_submission(model, test)
submission.head()

,building_id,damage_grade
0,122207000141,2
1,284909000871,5
2,241904000231,3
3,248308000891,3
4,300405002981,5


In [6]:
# Pengecekan format terhadap sample_submission
sample = load_data(SAMPLE_SUB_PATH)

print("Kolom sama      :", list(submission.columns) == list(sample.columns))
print("Jumlah baris    :", len(submission), "| sample:", len(sample))
print("ID sama         :", set(submission[ID_COL]) == set(sample[ID_COL]))
print("Nilai valid 1–5 :", submission[TARGET].isin([1, 2, 3, 4, 5]).all())
print("Missing value   :", submission.isna().sum().sum())

Kolom sama      : True
Jumlah baris    : 228629 | sample: 228629
ID sama         : True
Nilai valid 1–5 : True
Missing value   : 0


In [7]:
# Distribusi prediksi test dibanding distribusi target train
train_dist = load_data(TRAIN_PATH)[TARGET].value_counts(normalize=True)
train_dist.index = train_dist.index.str.replace("Grade ", "").astype(int)

pd.concat({"prediksi_test": submission[TARGET].value_counts(normalize=True),
           "train": train_dist}, axis=1).sort_index().mul(100).round(1)

,prediksi_test,train
damage_grade,,
1,9.0,10.3
2,15.2,11.4
3,19.0,17.9
4,24.7,24.1
5,32.2,36.2


In [8]:
# Submission versi 2
model_v2 = load_model(MODEL_DIR / "pipeline_model_v2.pkl")
sub_v2 = make_submission(model_v2, test, SUBMISSION_DIR / "submission_v2.csv")
print("Nilai valid:", sub_v2[TARGET].isin([1, 2, 3, 4, 5]).all(), "| baris:", len(sub_v2))
sub_v2[TARGET].value_counts(normalize=True).sort_index().mul(100).round(1)

Nilai valid: True | baris: 228629


damage_grade
1    10.6
2    13.2
3    18.7
4    25.5
5    32.0
Name: proportion, dtype: float64